# etcompare quickstart (Colab or local)

This notebook installs etcompare, writes the **synthetic** demonstration (or points at your own
configuration), runs it, and shows where the paired data, exclusions, metrics and report are.

* Every path is a variable you set below. Nothing assumes a drive letter or a Drive folder.
* The demonstration is generated with a fixed seed; its stations (`XX-SY1..3`) and models are not
  real, and every report page says so.
* Google Drive is optional: run the mount cell yourself only if your files are on Drive.

In [ ]:
# Install etcompare (Colab or any fresh environment). Skip this cell if it is installed.
# From a checkout or wheel you have, e.g.:  %pip install "/content/comparison[geo,plot]"
%pip install --quiet "etcompare[geo,plot] @ git+https://github.com/inkenbrandt/comparison"

## Optional: Google Drive

Run the next cell **yourself** only if your inputs are on Google Drive (Colab only). Remove the
`#` marks first. Nothing else in this notebook needs Drive.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

## Paths

`CONFIG` is your own configuration file, or `None` for the synthetic demonstration written to
`WORKDIR`. With `RASTERS = True` the demonstration also writes synthetic GeoTIFFs (about 1,850
small files) so raster indexing, extraction and representativeness run too (a few minutes).

In [ ]:
from pathlib import Path

WORKDIR = Path("etcompare_demo")   # e.g. Path("/content/drive/MyDrive/etcompare_demo")
CONFIG = None                      # e.g. Path("/content/drive/MyDrive/my_station/config.yaml")
RASTERS = False

In [ ]:
from etcompare.config import load_config
from etcompare.demo import write_demo

config_path = CONFIG or write_demo(WORKDIR, rasters=RASTERS)
cfg = load_config(config_path)
print("configuration:", config_path)
print("stations:", sorted(cfg.stations))
print("outputs go to:", cfg.output_dir)

## Run

The same as `etcompare run config.yaml` on the command line. A second run reuses every unit whose
inputs did not change.

In [ ]:
from etcompare.runner import run_stages

manifest, records = run_stages(cfg, ["validate", "index", "extract", "analyze", "report"],
                               command="notebook 01")
print(manifest.run_id, manifest.status)

In [ ]:
import pandas as pd

status = pd.DataFrame([r.row() for r in records])
status.groupby(["stage", "status"]).size()

## Paired data, exclusions and metrics

Every comparison writes its paired rows, the decision for every candidate day (kept or excluded,
with the reason) and its metrics. The metrics are plain functions of the paired rows.

In [ ]:
out = Path(cfg.output_dir)
cid = cfg.comparisons[0].comparison_id
paired = pd.read_parquet(out / "cohorts" / f"{cid}_paired.parquet")
membership = pd.read_parquet(out / "cohorts" / f"{cid}_membership.parquet")
print(cid, len(paired), "paired rows")
membership["reason_code"].fillna("included").value_counts()

In [ ]:
metrics = pd.read_parquet(next((out / "cohorts").glob(f"{cid}__*_metrics.parquet")))
row = metrics.query("metric == 'mae' and product == 'daily_pairs'").iloc[0]
one = paired[(paired.model == row.model) & (paired.site_id == row.scope)
             & (paired.reference_variant == row.reference_variant)
             & (paired.extraction_method == row.extraction_method)]
print(row.model, row.scope, "MAE from the table:", row.value, "recomputed:",
      (one.modeled_mm - one.observed_mm).abs().mean())

## Report

`report.html` is self-contained; `skipped_analyses.csv` lists every analysis that gave no result
and why. In Colab, download the HTML from the file browser.

In [ ]:
print(out / "report" / "report.html")
pd.read_csv(out / "report" / "skipped_analyses.csv").head(20)